In [ ]:

import asyncio
from datetime import date
import logging
from typing import List
from app.util.pubproc_token import get_token
from proclogic_api.app.schemas.publication_schemas import PublicationSchema
from proclogic_api.app.util.pubproc import process_publication_contract
from pydantic import TypeAdapter
import numpy as np
import httpx

from proclogic_api.app.config.postgres import get_session
from proclogic_api.app.config.settings import Settings


# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler("contract_backfill.log"), logging.StreamHandler()],
)
logger = logging.getLogger(__name__)

settings = Settings()


async def retrieve_publications(client: httpx.AsyncClient) -> None:
    """
    Main function that retrieves and processes publications.
    """
    with get_session() as session:
        pubproc_r = await get_timeframe_pubproc_search_data(client=client)
        pubproc_data = TypeAdapter(List[PublicationSchema]).validate_python(pubproc_r)

        # Process each publication
        for pub in pubproc_data:
            try:
                await process_publication_contract(client=client, pub=pub, session=session)
            except Exception as e:
                logging.error(
                    f"Error processing publication {pub.publication_workspace_id}: {e}"
                )


async def get_timeframe_pubproc_search_data(
    client: httpx.AsyncClient,
) -> dict:
    token = get_token()
    page_size = 100


    data = {
        "page": 1,
        "pageSize": page_size,
        "dispatchDateFrom": dispatch_date_from.strftime("%Y-%m-%d"),
        "dispatchDateTo": dispatch_date_to.strftime("%Y-%m-%d"),
    }

    headers = {
        "Authorization": f"Bearer {token}",
        "BelGov-Trace-Id": "2ce83af9-d524-43a6-8d1c-b19dff051aed",
    }

    r = await client.get(
        settings.pubproc_server + settings.path_sea_api + "/search/publications",
        params=data,
        headers=headers,
    )

    r_json = r.json()
    publications = r_json["publications"]
    total_count = int(r_json["totalCount"])

    if r.status_code == 200:
        pages = int(np.ceil(total_count / page_size))

        if pages > 1:
            for i in range(2, pages + 1):
                data["page"] = i
                r = await client.get(
                    settings.pubproc_server
                    + settings.path_sea_api
                    + "/search/publications",
                    params=data,
                    headers=headers,
                )
                await asyncio.sleep(5)
                publications.extend(r.json()["publications"])

    return publications


def main():
    async with httpx.AsyncClient() as client:
        retrieve_publications(client)


ModuleNotFoundError: No module named 'httpx'